# 02 — Train YOLOX-S on Colab
GPU work happens here. Weights, checkpoints, logs and run metadata persist in
Google Drive. Run cells deliberately; full training and weight downloading
have explicit switches. No Ultralytics package, CLI, trainer or weights.


## Repository and Drive
In Colab choose a GPU runtime with Python 3.10–3.12. Upload a ZIP containing
this repository's code, notebooks, configs, requirements and tests, and extract
it to `/content/aquafina-yolo-detector` using the Files pane. Do not include
datasets, checkpoints, caches, or `.git`. This works without a commit or push.
After an approved future push, cloning your own repository is also an option.
The repository on your computer remains the authoritative code copy.


In [ ]:
from pathlib import Path
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/aquafina-yolo-detector')
DRIVE = Path('/content/drive/MyDrive/aquafina-yolo')
assert (REPO / 'pyproject.toml').is_file(), 'Extract repository code first'
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'src'))
os.environ['PYTHONPATH'] = str(REPO / 'src') + ':/content/YOLOX'
PREPARED = DRIVE / 'processed/temple'


## Install the pinned GPU environment
This downloads official YOLOX **source** and Python dependencies only.
If Colab reports that already-imported packages changed, restart the session,
rerun the Repository and Drive cell, then continue with the audit cell.
Do not import torch/numpy before this install cell in a fresh session.


In [ ]:
subprocess.run([sys.executable, '-m', 'aquafina_detector.bootstrap', '--repo', str(REPO)], check=True)


In [ ]:
sys.path.insert(0, '/content/YOLOX')
import torch
from aquafina_detector.bootstrap import audit_runtime
from aquafina_detector.common import write_json
audit = audit_runtime()
assert torch.__version__.split('+')[0] == '2.5.1'
assert torch.cuda.is_available(), 'Select a GPU runtime'
write_json(DRIVE / 'environment/audit.json', audit)
print(torch.cuda.get_device_name(0), torch.__version__)


## Verify actual negative and mixed GPU batches
Synthetic fixtures test loading, Mosaic, class mapping and finite backward
passes. These are software tests, not a training dataset or accuracy evidence.


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', '-m', 'gpu'], check=True)


## Official pretrained weights — explicit download to Drive
Set DOWNLOAD_WEIGHTS only when ready. Existing weights are reused with their
provenance sidecar. Never use Ultralytics checkpoints or generic `yolo` commands.


In [ ]:
from aquafina_detector.bootstrap import download_pretrained
WEIGHTS = DRIVE / 'weights/pretrained/yolox_s.pth'
DOWNLOAD_WEIGHTS = False
if DOWNLOAD_WEIGHTS:
    download_pretrained(WEIGHTS)
assert WEIGHTS.is_file(), 'Enable the explicit download once, then reuse the weights'


## Dataset and checkpoint adaptation check
DATA defaults to prepared files on Drive. For faster reads, optionally copy
PREPARED to `/content/aquafina-data` using shutil.copytree and set DATA there.
Only temporary Colab storage may hold this copy; checkpoints still use Drive.


In [ ]:
DATA = PREPARED
from yolox.utils import load_ckpt
from aquafina_detector.experiment import AquafinaExp
original = torch.load(WEIGHTS, map_location='cpu', weights_only=False)['model']
model = AquafinaExp().get_model()
mismatched = [k for k, v in model.state_dict().items() if k in original and v.shape != original[k].shape]
assert mismatched and all('cls_preds' in k for k in mismatched), mismatched
load_ckpt(model, original)
assert model.head.num_classes == 1
del model, original
print('One-class adaptation verified:', mismatched)


## One-epoch smoke run, then resume the second epoch
Set RUN_SMOKE=True after preparing data. The two commands share a two-epoch
schedule; stopping after epoch one exercises a real checkpoint resume.
Use a new SMOKE name for a repeat. Check logs for finite losses.


In [ ]:
RUN_SMOKE = False
SMOKE = DRIVE / 'runs/smoke-v1'
if RUN_SMOKE:
    command = [sys.executable, '-m', 'aquafina_detector.train', '--data', str(DATA),
               '--run', str(SMOKE), '--batch-size', '2', '--epochs', '2']
    subprocess.run(command + ['--checkpoint', str(WEIGHTS), '--stop-after-epochs', '1'], check=True)
    saved = torch.load(SMOKE / 'latest_ckpt.pth', map_location='cpu', weights_only=False)
    assert saved['start_epoch'] == 1 and 'scaler' in saved and 'training_model' in saved
    del saved
    subprocess.run(command + ['--checkpoint', str(SMOKE / 'latest_ckpt.pth'), '--resume'], check=True)
    saved = torch.load(SMOKE / 'latest_ckpt.pth', map_location='cpu', weights_only=False)
    assert saved['start_epoch'] == 2
    del saved
    print('Checkpoint save and resume passed')


## Baseline
Set RUN_BASELINE=True after smoke verification. Start at batch 8; if CUDA runs
out of memory, use a NEW run name and batch 4, then 2. Learning rate scales
with batch size. Do not change batch/data/schedule while resuming.


In [ ]:
RUN_BASELINE = False
RUN = DRIVE / 'runs/baseline-v1'
BATCH = 8
if RUN_BASELINE:
    subprocess.run([sys.executable, '-m', 'aquafina_detector.train', '--data', str(DATA),
                    '--run', str(RUN), '--checkpoint', str(WEIGHTS), '--batch-size', str(BATCH)], check=True)


## Resume after disconnection
Remount Drive, restore repository/source and environment, then enable this
cell. Use the same RUN, BATCH and DATA as the original baseline. Checkpoints
are saved each epoch; work since the last completed epoch is lost.


In [ ]:
RESUME_BASELINE = False
if RESUME_BASELINE:
    subprocess.run([sys.executable, '-m', 'aquafina_detector.train', '--data', str(DATA),
                    '--run', str(RUN), '--checkpoint', str(RUN / 'latest_ckpt.pth'),
                    '--batch-size', str(BATCH), '--resume'], check=True)
